In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

print("=== AUGMENTATION COMPLÈTE DU DATASET SMD ===\n")

DATA_PATH = 'data/raw/SMD/train/'
PROCESSED_PATH = 'data/processed/'

os.makedirs(PROCESSED_PATH, exist_ok=True)

# Récupérer toutes les machines
train_files = sorted([f for f in os.listdir(DATA_PATH) if f.endswith('.txt')])
print(f"Nombre total de machines à traiter : {len(train_files)}\n")

def clean_and_load_file(file_path):
    """Nettoie et charge un fichier SMD"""
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    cleaned = [line.replace(',', ' ').strip() + '\n' for line in lines if line.strip()]
    
    temp_path = file_path + '.clean'
    with open(temp_path, 'w') as f:
        f.writelines(cleaned)
    
    df = pd.read_csv(temp_path, sep=r'\s+', header=None, engine='python')
    os.remove(temp_path)
    
    if df.shape[1] != 38:
        print(f"⚠️ {os.path.basename(file_path)} : {df.shape[1]} colonnes au lieu de 38")
        return None
    
    df.columns = [f'feature_{i}' for i in range(38)]
    return df.values  # Retourne en numpy array (plus rapide)

# Fonctions d'augmentation
def add_noise(data, factor=0.015):
    noise = np.random.normal(0, factor, data.shape)
    return np.clip(data + noise, 0.0, 1.0)

def random_scale(data, range=(0.96, 1.04)):
    scale = np.random.uniform(range[0], range[1], (data.shape[0], 1))
    return np.clip(data * scale, 0.0, 1.0)

def time_shift(data, max_shift=4):
    shifted = data.copy()
    for col in range(data.shape[1]):
        shift = np.random.randint(-max_shift, max_shift+1)
        if shift > 0:
            shifted[shift:, col] = data[:-shift, col]
        elif shift < 0:
            shifted[:shift, col] = data[-shift:, col]
    return shifted

# ====================== AUGMENTATION PRINCIPALE ======================

augmented_count = 0

for file_name in tqdm(train_files, desc="Traitement des machines"):
    file_path = os.path.join(DATA_PATH, file_name)
    machine_name = file_name.replace('.txt', '')
    
    data = clean_and_load_file(file_path)
    if data is None:
        continue
    
    # Sauvegarde des différentes versions
    np.save(os.path.join(PROCESSED_PATH, f'{machine_name}_original.npy'), data)
    np.save(os.path.join(PROCESSED_PATH, f'{machine_name}_noise.npy'), add_noise(data))
    np.save(os.path.join(PROCESSED_PATH, f'{machine_name}_scaled.npy'), random_scale(data))
    np.save(os.path.join(PROCESSED_PATH, f'{machine_name}_shift_noise.npy'), add_noise(time_shift(data)))
    
    augmented_count += 1

print("\n" + "="*60)
print(f"✓ AUGMENTATION TERMINÉE !")
print(f"Nombre de machines traitées : {augmented_count} / {len(train_files)}")
print(f"Nombre total de versions créées : {augmented_count * 4}")
print(f"Dossier de sauvegarde : {PROCESSED_PATH}")
print("="*60)

# Vérification
print("\nFichiers créés dans data/processed/ :")
files_created = os.listdir(PROCESSED_PATH)
print(f"Total fichiers : {len(files_created)}")
print("Exemples :", files_created[:10] if files_created else "Aucun")

# Sauvegarde d'un résumé
summary = pd.DataFrame({
    'machine': [f.replace('_original.npy', '') for f in files_created if f.endswith('_original.npy')],
    'points': [np.load(os.path.join(PROCESSED_PATH, f)).shape[0] for f in files_created if f.endswith('_original.npy')]
})
summary.to_csv(os.path.join(PROCESSED_PATH, 'augmentation_summary.csv'), index=False)
print(f"\nRésumé sauvegardé : data/processed/augmentation_summary.csv")

=== AUGMENTATION COMPLÈTE DU DATASET SMD ===

Nombre total de machines à traiter : 28



Traitement des machines: 100%|█████████████████| 28/28 [01:41<00:00,  3.62s/it]



✓ AUGMENTATION TERMINÉE !
Nombre de machines traitées : 28 / 28
Nombre total de versions créées : 112
Dossier de sauvegarde : data/processed/

Fichiers créés dans data/processed/ :
Total fichiers : 113
Exemples : ['machine-1-2_noise.npy', 'machine-1-2_shift_noise.npy', 'machine-2-4_scaled.npy', 'machine-3-7_shift_noise.npy', 'machine-2-3_shift_noise.npy', 'machine-2-6_noise.npy', 'machine-2-4_original.npy', 'machine-3-3_shift_noise.npy', 'machine-2-8_original.npy', 'machine-2-5_shift_noise.npy']

Résumé sauvegardé : data/processed/augmentation_summary.csv
